# RNN Implementation Notes

## RNN Input Shape

With `batch_first=True`, PyTorch expects:

`(batch_size, sequence_length, input_size)`

For example:

`x.shape = (4, 8, 10)`

* `4` → number of sequences/examples processed together
* `8` → number of time steps in each sequence
* `10` → number of features at each time step

---

## Hidden State

The hidden state stores the information the RNN carries from previous time steps.

Conceptually:

`h_t = tanh(W_x x_t + W_h h_(t-1) + b)`

So the current hidden state depends on:

* Current input `x_t`
* Previous hidden state `h_(t-1)`

### Hidden Size

`hidden_size = 20` means each hidden state is represented using 20 values.

Therefore, for each time step:

`input → 10 features`

`hidden state → 20 values`

The hidden state is passed to the next time step.

---

## `nn.RNN`

PyTorch provides the RNN through:

`nn.RNN(...)`

A basic RNN can be defined using:

* `input_size`
* `hidden_size`
* `num_layers`
* `batch_first=True`

The basic RNN uses `tanh` as its default nonlinearity.

When using `nn.RNN`, the recurrent calculation and `tanh` activation are handled internally.

---

## Number of Layers

`num_layers=2` means two RNN layers are stacked on top of each other.

It does **not** mean two time steps.

```text
Input sequence
      ↓
RNN Layer 1
      ↓
RNN Layer 2
      ↓
Output
```

---

## RNN Outputs

Calling:

`rnn_out, h_n = self.rnn(x)`

returns two tensors.

### `rnn_out`

Contains the hidden-state output at **every time step**.

For:

`x.shape = (4, 8, 10)`

and:

`hidden_size = 20`

the output shape is:

`rnn_out.shape = (4, 8, 20)`

Meaning:

* 4 sequences
* 8 time steps
* 20 hidden values at each time step

---

### `h_n`

Contains the **final hidden state** for each RNN layer.

Its shape depends on:

`(num_layers, batch_size, hidden_size)`

For a 2-layer RNN:

`(2, 4, 20)`

---

## Selecting the Final Time Step

To use the hidden representation from only the final time step:

`rnn_out[:, -1, :]`

This changes:

`(4, 8, 20) → (4, 20)`

So each sequence is now represented by one 20-dimensional hidden vector.

---

## Linear Output Layer

The final hidden representation can be passed into a linear layer:

`Linear(hidden_size, output_size)`

For:

`hidden_size = 20`

and:

`output_size = 5`

the transformation is:

`(4, 20) → (4, 5)`

The 5 values are the model's final outputs/predictions.

---

## Complete Flow

```text
Input
(4, 8, 10)
   ↓
RNN
hidden_size = 20
num_layers = 2
   ↓
RNN outputs
(4, 8, 20)
   ↓
Take final time step
(4, 20)
   ↓
Linear layer
20 → 5
   ↓
Prediction
(4, 5)
```

## Key Distinctions

* **Batch size** = how many sequences are processed simultaneously.
* **Sequence length** = how many time steps each sequence contains.
* **Input size** = how many features each time step contains.
* **Hidden size** = how many values represent the RNN's hidden state.
* **Number of layers** = how many RNN layers are stacked.
* **`rnn_out`** = hidden outputs from every time step.
* **`h_n`** = final hidden state of each RNN layer.


In [1]:
import torch
import torch.nn as nn

In [2]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size=input_size,
                          hidden_size=hidden_size,
                          num_layers=num_layers,
                          batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        rnn_out, h_n = self.rnn(x)
        last_output = rnn_out[:, -1, :]
        output = self.fc(last_output)

        return output

In [4]:
model = RNNModel(input_size=10, hidden_size=20, output_size=5, num_layers=2)
print("RNN Model:")
print(model)
x = torch.randn(4, 8, 10)
y = torch.randn(4,5)
output = model(x)
criterion = nn.MSELoss()
loss = criterion(output, y)
print("Output shape:", output.shape)
print("Loss: ",loss)

RNN Model:
RNNModel(
  (rnn): RNN(10, 20, num_layers=2, batch_first=True)
  (fc): Linear(in_features=20, out_features=5, bias=True)
)
Output shape: torch.Size([4, 5])
Loss:  tensor(1.1484, grad_fn=<MseLossBackward0>)
